# Group Assignment (Demo 2)
This notebook contains introductory Machine Learning content for the Group Assignment, linked to Big Data session 3: Machine Learning. 

In what follows, we will:
1. Extract & engineer new features on the numerical stocks dataset.
2. Train predictive models.
3. Predict values for unseen cases and evaluate them on a hold-out test set. 

### 1. Initialize PySpark & import packages

In [1]:
# import findspark
import findspark
# initialize findspark with spark directory
findspark.init(r"C:\spark\spark-4.0.1-bin-hadoop3")
# import pyspark
import pyspark
# create spark contextp
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [2]:
# inspect spark session
spark

In [3]:
# Install required packages for Demo 2
!pip install seaborn matplotlib

# Now import everything
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Import pyspark functionalities
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *

print("All packages imported successfully!")

# import pyspark functionalities
from pyspark.sql.types import NumericType
from pyspark.ml import Pipeline
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# turn of warnings
import warnings
warnings.filterwarnings("ignore")


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


All packages imported successfully!


### 2. Additional Data Cleaning

In [29]:
# Load the monthly partitioned data (this should have all your final processing)
monthly_partitioned_path = r"C:\Users\Beheerder\Documents\HIR\Big Data\Groupwork\Big_data_group_2\Cleaned_data\monthly_partitioned_stock_data"

df_cgc = spark.read.parquet(monthly_partitioned_path)

print("Monthly partitioned data loaded successfully!")
df_cgc.show(10)

Monthly partitioned data loaded successfully!
+------+-------------------+-------------------+-----+--------------------+--------------------+----+----+----+------+----+-----+
|ticker|   Local Date (CET)|    Local Date (ET)|Close|                 Net|                %Chg|Open|High| Low|Volume|Year|Month|
+------+-------------------+-------------------+-----+--------------------+--------------------+----+----+----+------+----+-----+
|   EQX|2025-08-30 01:56:00|2025-08-29 19:56:00| 8.78|                 0.0|                 0.0|8.78|8.78|8.78|  17.0|2025|    8|
|   EQX|2025-08-30 01:52:00|2025-08-29 19:52:00| 8.78| 0.00999999999999979| 0.00114025085518812|8.78|8.78|8.78|1000.0|2025|    8|
|   EQX|2025-08-30 01:21:00|2025-08-29 19:21:00| 8.77|  0.0999999999999996|  0.0115340253748558|8.75|8.77|8.75| 152.0|2025|    8|
|   EQX|2025-08-30 01:03:00|2025-08-29 19:03:00| 8.67| -0.0800000000000001|-0.00914285714285715|8.67|8.67|8.67| 100.0|2025|    8|
|   EQX|2025-08-30 01:02:00|2025-08-29 19:02

In [30]:
# Drop second date column (CET time, we do not need this anymore):
df_cgc = df_cgc.drop("Local Date (CET)")

In [31]:
# Drop duplicates based on date column
df_cgc = df_cgc.dropDuplicates(["Local Date (ET)"])

In [32]:
# Load additional company data
# Path to your announcements folder
announcements_dir = r"C:\Users\Beheerder\Documents\HIR\Big Data\Groupwork\Big_data_group_2\Data\Dependent_data"

# Load all CSV files in that directory
announcements_all = spark.read.csv(announcements_dir + "/*.csv", header=True, inferSchema=True)

print(f"Loaded announcements for all companies")
announcements_all.show(10)

Loaded announcements for all companies
+--------------------+------+----------+--------------------+
|          stock_name|ticker| date-time|                type|
+--------------------+------+----------+--------------------+
|Dolly Varden Silv...|    DV|2024-01-16| exploration_results|
|Dolly Varden Silv...|    DV|2024-02-12| exploration_results|
|Dolly Varden Silv...|    DV|2024-03-04|financing_announc...|
|Dolly Varden Silv...|    DV|2024-06-19| exploration_results|
|Dolly Varden Silv...|    DV|2024-08-12| exploration_results|
|Dolly Varden Silv...|    DV|2024-08-19|financing_announc...|
|Dolly Varden Silv...|    DV|2024-09-04|financing_announc...|
|Dolly Varden Silv...|    DV|2024-09-09| exploration_results|
|Dolly Varden Silv...|    DV|2024-09-27|financing_announc...|
|Dolly Varden Silv...|    DV|2024-11-04| exploration_results|
+--------------------+------+----------+--------------------+
only showing top 10 rows


In [33]:
# Check the schema of your announcements data
print("Announcements schema:")
announcements_all.printSchema()

# Show some sample data with all columns
print("Sample announcements data:")
announcements_all.show(10, truncate=False)

Announcements schema:
root
 |-- stock_name: string (nullable = true)
 |-- ticker: string (nullable = true)
 |-- date-time: date (nullable = true)
 |-- type: string (nullable = true)

Sample announcements data:
+------------------------+------+----------+----------------------+
|stock_name              |ticker|date-time |type                  |
+------------------------+------+----------+----------------------+
|Dolly Varden Silver Corp|DV    |2024-01-16|exploration_results   |
|Dolly Varden Silver Corp|DV    |2024-02-12|exploration_results   |
|Dolly Varden Silver Corp|DV    |2024-03-04|financing_announcement|
|Dolly Varden Silver Corp|DV    |2024-06-19|exploration_results   |
|Dolly Varden Silver Corp|DV    |2024-08-12|exploration_results   |
|Dolly Varden Silver Corp|DV    |2024-08-19|financing_announcement|
|Dolly Varden Silver Corp|DV    |2024-09-04|financing_announcement|
|Dolly Varden Silver Corp|DV    |2024-09-09|exploration_results   |
|Dolly Varden Silver Corp|DV    |2024-09-2

In [34]:
# First, let's see what announcement types we have across ALL companies
print("\nAnnouncement distribution for ALL companies:")
announcements_all.groupBy("type").count().orderBy(F.desc("count")).show(truncate=False)

# Also see the distribution by company
print("\nAnnouncement count by company:")
announcements_all.groupBy("ticker").count().orderBy(F.desc("count")).show(truncate=False)

# See announcement types for each company
print("\nAnnouncement types by company:")
announcements_all.groupBy("ticker", "type").count().orderBy("ticker", F.desc("count")).show(truncate=False)


Announcement distribution for ALL companies:
+----------------------+-----+
|type                  |count|
+----------------------+-----+
|exploration_results   |128  |
|financing_announcement|61   |
|closing_announcement  |32   |
|property_acquisition  |18   |
|other                 |1    |
+----------------------+-----+


Announcement count by company:
+------+-----+
|ticker|count|
+------+-----+
|DV    |25   |
|FNM   |23   |
|ELO   |23   |
|AYA   |23   |
|FDY   |22   |
|FVL   |19   |
|FURY  |18   |
|FOM   |18   |
|FAS   |14   |
|EDG   |13   |
|CGC   |12   |
|DMX   |10   |
|CKG   |5    |
|ESK   |4    |
|EQX   |4    |
|CG    |3    |
|EXMPL |3    |
|FINX  |1    |
+------+-----+


Announcement types by company:
+------+----------------------+-----+
|ticker|type                  |count|
+------+----------------------+-----+
|AYA   |exploration_results   |17   |
|AYA   |financing_announcement|6    |
|CG    |exploration_results   |2    |
|CG    |financing_announcement|1    |
|CGC   |explo

### 1. Feature Engineering
In order to maximize the predictive power of a dataset, it is important to capture 'hidden' features that can be discovered by transforming or combining base features. We're predicting financing announcements on a biweekly basis, so we will engineer features that capture stock price and volume dynamics over the 2 previous weeks as well as longer-term trends.

#### 1.1. Create 2-week period identifier


In [35]:
# Get the minimum date to use as reference point
min_date = df_cgc.agg(F.min("Local Date (ET)")).collect()[0][0]

# Calculate days since the start date, then divide by 14 to get 2-week periods
df_cgc = df_cgc.withColumn("days_since_start", 
                           F.datediff(F.col("Local Date (ET)"), F.lit(min_date)))

df_cgc = df_cgc.withColumn("biweekly_period", 
                           F.concat(F.lit("P"), 
                                   F.floor(F.col("days_since_start") / 14).cast("string")))

# Optional: Keep year column for reference
df_cgc = df_cgc.withColumn("year", F.year("Local Date (ET)"))

# ADD THE MISSING 'Date_only' COLUMN
# The 'Local Date (ET)' column contains a timestamp (date and time).
# We must convert it to a Date type to group by day.
# This step prevents the "Column 'Date_only' is ambiguous" error.
df_cgc = df_cgc.withColumn(
    "Date_only", 
    F.to_date(F.col("Local Date (ET)"))
)

df_cgc.show(20)

+------+-------------------+-----+------------------+--------------------+----+-----+----+------+----+-----+----------------+---------------+----------+
|ticker|    Local Date (ET)|Close|               Net|                %Chg|Open| High| Low|Volume|year|Month|days_since_start|biweekly_period| Date_only|
+------+-------------------+-----+------------------+--------------------+----+-----+----+------+----+-----+----------------+---------------+----------+
|   ELO|2024-01-15 10:06:00| 1.88|             -0.01| -0.0052910052910053|1.88| 1.88|1.88| 500.0|2024|    1|               0|             P0|2024-01-15|
|   ELO|2024-01-15 10:07:00| 1.88|               0.0|                 0.0|1.88| 1.88|1.88| 400.0|2024|    1|               0|             P0|2024-01-15|
|   ARG|2024-01-15 11:00:00| 1.36|               0.0|                 0.0|1.36| 1.36|1.36|2000.0|2024|    1|               0|             P0|2024-01-15|
|   ARG|2024-01-15 12:04:00| 1.35|             -0.01|-0.00735294117647059|1.35| 1.

In [36]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

# NOTE: Assuming df_cgc has already been loaded and has the columns 'Date_only',
# 'biweekly_period', and 'year' defined correctly.

# Step 1: Just do the daily aggregation and see if it works
daily_data_simple = df_cgc.groupBy(
    F.col("ticker"),  # ADDED: Group by ticker
    F.col("Date_only"), 
    F.col("biweekly_period"), 
    F.col("year")
).agg(
    F.first("Open").alias("daily_open"),
    F.last("Close").alias("daily_close"),
    F.max("High").alias("daily_high"),
    F.min("Low").alias("daily_low"),
    F.sum("Volume").alias("daily_volume")
)

print("Simple daily aggregation:")
daily_data_simple.show(10)

# If that works, then add calculations one by one
# Using .count() here will trigger another Spark job, but it serves as a safety check.
if daily_data_simple.count() > 0:
    # Add daily_return
    # CRITICAL FIX: Cast the columns to DoubleType to allow numerical division.
    daily_data_step1 = daily_data_simple.withColumn("daily_return", 
        (F.col("daily_close").cast(DoubleType()) - F.col("daily_open").cast(DoubleType())) / F.col("daily_open").cast(DoubleType())
    )
    print("Added daily_return:")
    daily_data_step1.show(10)

Simple daily aggregation:
+------+----------+---------------+----+----------+-----------+----------+---------+------------+
|ticker| Date_only|biweekly_period|year|daily_open|daily_close|daily_high|daily_low|daily_volume|
+------+----------+---------------+----+----------+-----------+----------+---------+------------+
|   ARG|2024-01-15|             P0|2024|      1.36|       1.36|      1.38|     1.35|     71200.0|
|   ARG|2024-01-16|             P0|2024|      1.34|       1.33|      1.35|     1.32|    140200.0|
|   ARG|2024-01-17|             P0|2024|      1.32|       1.29|      1.32|     1.28|    292600.0|
|   ARG|2024-01-18|             P0|2024|      1.28|      1.285|      1.32|     1.27|    119500.0|
|   ARG|2024-01-19|             P0|2024|      1.29|        1.3|      1.32|     1.28|    123300.0|
|   ARG|2024-01-22|             P0|2024|      1.32|       1.33|      1.34|     1.29|    155000.0|
|   ARG|2024-01-23|             P0|2024|      1.35|        1.3|      1.36|      1.3|    1070

In [37]:
# Check the data types of the aggregated columns
print("Daily data schema:")
daily_data_simple.printSchema()

# Convert the numeric columns to double type
daily_data_fixed = daily_data_simple.select(
    "ticker",  # Added 'ticker' column
    "Date_only",
    "biweekly_period", 
    "year",
    F.col("daily_open").cast("double").alias("daily_open"),
    F.col("daily_close").cast("double").alias("daily_close"),
    F.col("daily_high").cast("double").alias("daily_high"),
    F.col("daily_low").cast("double").alias("daily_low"),
    F.col("daily_volume").cast("double").alias("daily_volume")
)

print("Fixed schema:")
daily_data_fixed.printSchema()

# Now try the calculations
daily_data_step1 = daily_data_fixed.withColumn("daily_return", 
                                               (F.col("daily_close") - F.col("daily_open")) / F.col("daily_open"))

print("Added daily_return:")
daily_data_step1.show(10)

Daily data schema:
root
 |-- ticker: string (nullable = true)
 |-- Date_only: date (nullable = true)
 |-- biweekly_period: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- daily_open: string (nullable = true)
 |-- daily_close: string (nullable = true)
 |-- daily_high: string (nullable = true)
 |-- daily_low: string (nullable = true)
 |-- daily_volume: double (nullable = true)

Fixed schema:
root
 |-- ticker: string (nullable = true)
 |-- Date_only: date (nullable = true)
 |-- biweekly_period: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- daily_open: double (nullable = true)
 |-- daily_close: double (nullable = true)
 |-- daily_high: double (nullable = true)
 |-- daily_low: double (nullable = true)
 |-- daily_volume: double (nullable = true)

Added daily_return:
+------+----------+---------------+----+----------+-----------+----------+---------+------------+--------------------+
|ticker| Date_only|biweekly_period|year|daily_open|daily_close|daily_h

In [38]:
# Continue with the remaining calculations
daily_data_final = daily_data_step1.withColumn("daily_range", 
                                   (F.col("daily_high") - F.col("daily_low")) / F.col("daily_close"))

# Add window for previous close
# CRUCIAL Adjustment: PARTITION BY 'ticker'
window_daily = Window.partitionBy("ticker").orderBy("Date_only")

daily_data_final = daily_data_final.withColumn("prev_close", 
                                   F.lag("daily_close", 1).over(window_daily))

daily_data_final = daily_data_final.withColumn("gap", 
                                   (F.col("daily_open") - F.col("prev_close")) / F.col("prev_close"))

print(f"Total trading days: {daily_data_final.count()}")
daily_data_final.select("ticker", "Date_only", "biweekly_period", "daily_open", "daily_close", "daily_high", "daily_low", "daily_range", "gap",
                  "daily_volume", "daily_return").show(10, truncate=False)

Total trading days: 1227
+------+----------+---------------+----------+-----------+----------+---------+--------------------+----------------------+------------+---------------------+
|ticker|Date_only |biweekly_period|daily_open|daily_close|daily_high|daily_low|daily_range         |gap                   |daily_volume|daily_return         |
+------+----------+---------------+----------+-----------+----------+---------+--------------------+----------------------+------------+---------------------+
|ARG   |2024-01-15|P0             |1.36      |1.36       |1.38      |1.35     |0.02205882352941162 |NULL                  |71200.0     |0.0                  |
|ARG   |2024-01-16|P0             |1.34      |1.33       |1.35      |1.32     |0.02255639097744363 |-0.014705882352941188 |140200.0    |-0.007462686567164185|
|ARG   |2024-01-17|P0             |1.32      |1.29       |1.32      |1.28     |0.031007751937984523|-0.0075187969924812095|292600.0    |-0.022727272727272745|
|ARG   |2024-01-18|P0


#### 1.2 Aggregate to daily level features
For proper volume analysis, we need to aggregate minute-level data to daily data first. This ensures volume metrics represent actual daily trading volume, not minute averages.


In [39]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType # CRITICAL: Import DoubleType for casting

# --- AGGREGATION (WORKS, BUT TYPES MIGHT BE IMPRECISE) ---

# Create date column (without time) for daily aggregation
df_cgc = df_cgc.withColumn("Date_only", F.to_date("Local Date (ET)"))

# Aggregate to daily level
daily_data = df_cgc.groupBy("ticker", "Date_only", "biweekly_period", "year").agg(
    F.first("Open").alias("daily_open"),
    F.last("Close").alias("daily_close"),
    F.max("High").alias("daily_high"),
    F.min("Low").alias("daily_low"),
    F.sum("Volume").alias("daily_volume"),
    F.max("Volume").alias("daily_max_volume")
)

# --- CALCULATIONS (FIXED WITH CASTING) ---

# Define the Window for time series calculations
window_daily = Window.partitionBy("ticker").orderBy("Date_only")

# 1. Calculate daily_return (Requires casting to DoubleType)
daily_data = daily_data.withColumn("daily_return", 
    (F.col("daily_close").cast(DoubleType()) - F.col("daily_open").cast(DoubleType())) / F.col("daily_open").cast(DoubleType())
)

# 2. Calculate daily_range (Requires casting to DoubleType)
daily_data = daily_data.withColumn("daily_range", 
    (F.col("daily_high").cast(DoubleType()) - F.col("daily_low").cast(DoubleType())) / F.col("daily_close").cast(DoubleType())
)

# 3. Calculate prev_close (Lag doesn't require casting, but will use the type of daily_close)
daily_data = daily_data.withColumn("prev_close", 
    F.lag("daily_close", 1).over(window_daily)
)

# 4. Calculate gap (Requires casting to DoubleType, especially as prev_close comes from lag)
daily_data = daily_data.withColumn("gap", 
    (F.col("daily_open").cast(DoubleType()) - F.col("prev_close").cast(DoubleType())) / F.col("prev_close").cast(DoubleType())
)

# --- OUTPUT ---

print(f"Total trading days: {daily_data.count()}")
daily_data.select("ticker", "Date_only", "biweekly_period", "daily_open", "daily_close", "daily_high", "daily_low", "daily_range", "gap",
                  "daily_volume", "daily_max_volume", "daily_return").show(10, truncate=False)

Total trading days: 1227
+------+----------+---------------+----------+-----------+----------+---------+--------------------+----------------------+------------+----------------+---------------------+
|ticker|Date_only |biweekly_period|daily_open|daily_close|daily_high|daily_low|daily_range         |gap                   |daily_volume|daily_max_volume|daily_return         |
+------+----------+---------------+----------+-----------+----------+---------+--------------------+----------------------+------------+----------------+---------------------+
|ARG   |2024-01-15|P0             |1.36      |1.36       |1.38      |1.35     |0.02205882352941162 |NULL                  |71200.0     |8600.0          |0.0                  |
|ARG   |2024-01-16|P0             |1.34      |1.33       |1.35      |1.32     |0.02255639097744363 |-0.014705882352941188 |140200.0    |8600.0          |-0.007462686567164185|
|ARG   |2024-01-17|P0             |1.32      |1.29       |1.32      |1.28     |0.03100775193798

Now the data is aggregated to daily level, we can add the company announcements.

In [40]:
# Join announcements with price data
# Select all columns from daily_data and the 'type' column from announcements
daily_data_with_announcements = daily_data_final.join(
    announcements_all, 
    (daily_data_final.Date_only == announcements_all["date-time"]) & \
    (daily_data_final.ticker == announcements_all["ticker"]),    how='left'
).select(
    daily_data_final["*"],
    announcements_all["type"].alias("announcement_type")
)

print("Data joined with announcements:")
daily_data_with_announcements.show(20, truncate=False)

# Check how many days have announcements
print("\nDays with announcements:")
days_with_announcements = daily_data_with_announcements.filter(F.col("announcement_type").isNotNull())
print(f"Total days with announcements: {days_with_announcements.count()}")

# See the distribution of announcement types
print("\nAnnouncement types in the joined data:")
daily_data_with_announcements.groupBy("announcement_type").count().orderBy(F.desc("count")).show(truncate=False)

Data joined with announcements:
+------+----------+---------------+----+----------+-----------+----------+---------+------------+---------------------+--------------------+----------+----------------------+-----------------+
|ticker|Date_only |biweekly_period|year|daily_open|daily_close|daily_high|daily_low|daily_volume|daily_return         |daily_range         |prev_close|gap                   |announcement_type|
+------+----------+---------------+----+----------+-----------+----------+---------+------------+---------------------+--------------------+----------+----------------------+-----------------+
|ARG   |2024-01-15|P0             |2024|1.36      |1.36       |1.38      |1.35     |71200.0     |0.0                  |0.02205882352941162 |NULL      |NULL                  |NULL             |
|ARG   |2024-01-16|P0             |2024|1.34      |1.33       |1.35      |1.32     |140200.0    |-0.007462686567164185|0.02255639097744363 |1.36      |-0.014705882352941188 |NULL             |
|AR

In [41]:
daily_data_with_announcements.show()

+------+----------+---------------+----+----------+-----------+----------+---------+------------+--------------------+--------------------+----------+--------------------+-----------------+
|ticker| Date_only|biweekly_period|year|daily_open|daily_close|daily_high|daily_low|daily_volume|        daily_return|         daily_range|prev_close|                 gap|announcement_type|
+------+----------+---------------+----+----------+-----------+----------+---------+------------+--------------------+--------------------+----------+--------------------+-----------------+
|   ARG|2024-01-15|             P0|2024|      1.36|       1.36|      1.38|     1.35|     71200.0|                 0.0| 0.02205882352941162|      NULL|                NULL|             NULL|
|   ARG|2024-01-16|             P0|2024|      1.34|       1.33|      1.35|     1.32|    140200.0|-0.00746268656716...| 0.02255639097744363|      1.36|-0.01470588235294...|             NULL|
|   ARG|2024-01-17|             P0|2024|      1.32

#### 1.3. Biweekly feature aggregation
Now that we have daily-level data, we can aggregate to biweekly level to create our features.

In [42]:
# Now aggregate the DAILY data to 2-week periods
# **CRITICAL CHANGE**: Group by both 'ticker' and 'biweekly_period'
biweekly_data = daily_data_with_announcements.groupBy("ticker", "biweekly_period").agg(
    # Time identifiers
    F.min("Date_only").alias("period_start"),
    F.max("Date_only").alias("period_end"),
    
    # PRICE FEATURES
    F.first("daily_open").alias("period_open"),
    F.last("daily_close").alias("period_close"),
    F.max("daily_high").alias("period_high"),
    F.min("daily_low").alias("period_low"),
    F.avg("daily_close").alias("avg_close"),
    
    # Price momentum within period
    ((F.last("daily_close") - F.first("daily_open")) / F.first("daily_open")).alias("period_return"),

    # VOLATILITY FEATURES
    F.stddev("daily_close").alias("price_volatility"),
    F.stddev("daily_return").alias("return_volatility"),
    
    # Intraday volatility
    F.avg("daily_range").alias("avg_intraday_range"),
    F.max("daily_range").alias("max_intraday_range"),
    
    # VOLUME FEATURES
    F.sum("daily_volume").alias("total_volume"),
    F.avg("daily_volume").alias("avg_daily_volume"),
    F.max("daily_volume").alias("max_daily_volume"),
    F.stddev("daily_volume").alias("volume_volatility"),
    
    # Volume concentration (max daily volume / avg daily volume)
    (F.max("daily_volume") / F.avg("daily_volume")).alias("volume_concentration"),
    
    # PRICE-VOLUME RELATIONSHIP
    
    # Average volume on up days vs down days
    F.avg(F.when(F.col("daily_return") > 0, F.col("daily_volume"))).alias("avg_volume_up_days"),
    F.avg(F.when(F.col("daily_return") < 0, F.col("daily_volume"))).alias("avg_volume_down_days"),
    
    # GAP FEATURES 
    # Count of significant gaps (>2%)
    F.sum(F.when(F.abs(F.col("gap")) > 0.02, 1).otherwise(0)).alias("gap_count"),
    
    # Average gap size
    F.avg(F.abs(F.col("gap"))).alias("avg_gap_size"),
    
    # Max gap (largest overnight gap)
    F.max(F.abs(F.col("gap"))).alias("max_gap"),
        
    # MOMENTUM INDICATORS
    # Count of up days
    F.sum(F.when(F.col("daily_return") > 0, 1).otherwise(0)).alias("up_days"),
    
    # Count of down days
    F.sum(F.when(F.col("daily_return") < 0, 1).otherwise(0)).alias("down_days"),
        
    # ANNOUNCEMENT FEATURES (counts)
    F.sum(F.when(F.col("announcement_type") == "property_acquisition", 1).otherwise(0)).alias("property_acquisition"),
    F.sum(F.when(F.col("announcement_type") == "exploration_results", 1).otherwise(0)).alias("exploration_results"),

    # DEPENDENT VARIABLE (binary)
    F.max(F.when(F.col("announcement_type") == "financing_announcement", 1).otherwise(0)).alias("financing_announcement"),
)

biweekly_data.show()

+------+---------------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+--------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+--------------------+-------------------+----------------------+
|ticker|biweekly_period|period_start|period_end|period_open|period_close|period_high|period_low|         avg_close|       period_return|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range|total_volume|  avg_daily_volume|max_daily_volume| volume_volatility|volume_concentration|avg_volume_up_days|avg_volume_down_days|gap_count|        avg_gap_size|             max_gap|up_days|down_days|property_acquisition|exploration_results|financing_announcement|
+------+---------------+--------

In [43]:
# Price deviation from previous year average
# Previous 365 days = 26 periods (excluding current)
window_year = Window.partitionBy("ticker").orderBy("period_start").rowsBetween(-26, -1)

# Calculate rolling year average close price
biweekly_data = biweekly_data.withColumn(
    "rolling_year_avg_close", 
    F.avg("avg_close").over(window_year)
)

# Calculate percentage difference from year average
biweekly_data = biweekly_data.withColumn(
    "price_deviation_from_year", 
    (F.col("avg_close") - F.coalesce(F.col("rolling_year_avg_close"), F.col("avg_close"))) / 
    F.coalesce(F.col("rolling_year_avg_close"), F.col("avg_close"))
)
biweekly_data = biweekly_data.drop("rolling_year_avg_close")
biweekly_data.show()

+------+---------------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+--------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+--------------------+-------------------+----------------------+-------------------------+
|ticker|biweekly_period|period_start|period_end|period_open|period_close|period_high|period_low|         avg_close|       period_return|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range|total_volume|  avg_daily_volume|max_daily_volume| volume_volatility|volume_concentration|avg_volume_up_days|avg_volume_down_days|gap_count|        avg_gap_size|             max_gap|up_days|down_days|property_acquisition|exploration_results|financing_announcement|price_d

#### 1.4. Lagged features
Here we create lagged features, meaning that features from the previous period can also be used in the current period, to predict the financing announcement in the next period. In other words, we are using information from 4 weeks to predict the next 2 weeks. You can experiment with different lags or compare the current data with long term trends.

In [44]:
window_period = Window.partitionBy("ticker").orderBy("period_start")

# Add features from previous 2-week period
biweekly_data = biweekly_data.withColumn("prev_period_return", F.lag("period_return", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_price_volatility", F.lag("price_volatility", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_avg_daily_volume", F.lag("avg_daily_volume", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_volume_volatility", F.lag("volume_volatility", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_gap_count", F.lag("gap_count", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_exploration_results", F.lag("exploration_results", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_property_acquisition", F.lag("property_acquisition", 1).over(window_period))

# Add autoregressive features (BE CAUTIOUS WITH DATA LEAKAGE)
# We use data of the last 4 weeks to predict the next 2 weeks, which is a correct setup. 
# You can experiment with even longer lags, it would make sense that a company doesn't announce financings in multiple consecutive periods. 
biweekly_data = biweekly_data.withColumn("current_financing_announcement", F.col("financing_announcement")) # current period
biweekly_data = biweekly_data.withColumn("prev_financing_announcement", F.lag("financing_announcement", 1).over(window_period)) # previous period

# Step 4: Create comparative features (current vs previous period)
biweekly_data = biweekly_data.withColumn("return_change", 
                                                 F.col("period_return") - F.col("prev_period_return"))
biweekly_data = biweekly_data.withColumn("volatility_change", 
                                                 F.col("price_volatility") / (F.col("prev_price_volatility") + 1e-06))
biweekly_data = biweekly_data.withColumn("volume_change", 
                                                 F.col("avg_daily_volume") / (F.col("prev_avg_daily_volume") + 1e-06))

# Sort by period start date
biweekly_data = biweekly_data.orderBy("period_start")

In [45]:
biweekly_data.show()

+------+---------------+------------+----------+-----------+------------+-----------+----------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+--------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+--------------------+-------------------+----------------------+-------------------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+--------------------+------------------+-------------------+
|ticker|biweekly_period|period_start|period_end|period_open|period_close|period_high|period_low|          avg_close|       period_return|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_r

#### 1.5. Dependent Variable

The ``next_announcement`` label is created by looking at whether there was a financing announcement in the *next* biweekly period. We move the announcement information back by one period to align it with the features from the previous period.

In [46]:
# define a window to access the previous period
# **CRITICAL CHANGE**: Partition by 'ticker' to ensure the lead function looks at the next period of the SAME STOCK.
window_spec = Window.partitionBy("ticker").orderBy("period_start")

# create the 'next_announcement' label
biweekly_data = biweekly_data.withColumn("next_announcement", F.lead("financing_announcement").over(window_spec)).drop("financing_announcement")

# drop the last period as we don't have dependent information for it
biweekly_data = biweekly_data.filter(biweekly_data.next_announcement.isNotNull())

# show
biweekly_data.show(50)

+------+---------------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+--------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+--------------------+-------------------+-------------------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+--------------------+-------------------+--------------------+-----------------+
|ticker|biweekly_period|period_start|period_end|period_open|period_close|period_high|period_low|         avg_close|       period_return|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range|

In [47]:
biweekly_data.printSchema()

root
 |-- ticker: string (nullable = true)
 |-- biweekly_period: string (nullable = true)
 |-- period_start: date (nullable = true)
 |-- period_end: date (nullable = true)
 |-- period_open: double (nullable = true)
 |-- period_close: double (nullable = true)
 |-- period_high: double (nullable = true)
 |-- period_low: double (nullable = true)
 |-- avg_close: double (nullable = true)
 |-- period_return: double (nullable = true)
 |-- price_volatility: double (nullable = true)
 |-- return_volatility: double (nullable = true)
 |-- avg_intraday_range: double (nullable = true)
 |-- max_intraday_range: double (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- avg_daily_volume: double (nullable = true)
 |-- max_daily_volume: double (nullable = true)
 |-- volume_volatility: double (nullable = true)
 |-- volume_concentration: double (nullable = true)
 |-- avg_volume_up_days: double (nullable = true)
 |-- avg_volume_down_days: double (nullable = true)
 |-- gap_count: long (nullable

In [48]:
# Write the final dataset to parquet
biweekly_data_path = r"C:\Users\Beheerder\Documents\HIR\Big Data\Groupwork\Big_data_group_2\Cleaned_data\biweekly_data.parquet"

biweekly_data.write.mode("overwrite").parquet(biweekly_data_path)

print(f"✅ Biweekly dataset saved to: {biweekly_data_path}")
print(f"Total biweekly periods: {biweekly_data.count()}")

✅ Biweekly dataset saved to: C:\Users\Beheerder\Documents\HIR\Big Data\Groupwork\Big_data_group_2\Cleaned_data\biweekly_data.parquet
Total biweekly periods: 232


## 2. Modeling

In this section, we’ll train some basic predictive models to predict financing announcements. We’ll use the first 8 months of our dataset for training, reserving the final 3 months for model evaluation. By relying exclusively on lagged variables as predictors, we ensure that the model only uses historical information to make forward-looking predictions—mirroring the real-world constraint of forecasting based solely on past data.

When working with time series data, particularly in stock price analysis, it’s essential to maintain the temporal order of observations. Randomly splitting the data into training and testing sets can lead to information leakage, as future information may inadvertently influence the model during training. To prevent this, a time-based split should be used instead.

### 2.1 Time-based Split
A time-based split involves using the earlier portion of the dataset for training and reserving the later portion for testing. This approach mirrors a real-world scenario in which we predict future announcements based solely on historical data, ensuring that our model remains grounded in past information while evaluating its ability to forecast what comes next.

In [49]:
# Use the first 2/3 of the data for training and the last 1/3 for testing
train_size = int(biweekly_data.count() * 0.67)
train_df = biweekly_data.limit(train_size)

# Use the remaining data for testing
test_df = biweekly_data.subtract(train_df)

In [50]:
# Because we are only working with a small dataset using one company, we want to make sure both train and test contain the positive class
print("Training data:")
train_df.groupBy('next_announcement').count().show()
print("Test data:")
test_df.groupBy('next_announcement').count().show()

Training data:
+-----------------+-----+
|next_announcement|count|
+-----------------+-----+
|                0|  147|
|                1|    8|
+-----------------+-----+

Test data:
+-----------------+-----+
|next_announcement|count|
+-----------------+-----+
|                1|    1|
|                0|   76|
+-----------------+-----+



### 2.2. Check Missing Values

In [51]:
# get all numeric columns
numeric_cols = [c for c, t in train_df.dtypes if isinstance(train_df.schema[c].dataType, NumericType)]

# count null values in each column
null_counts_train = train_df.select([F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in numeric_cols])
null_counts_train.show()

null_counts_test = test_df.select([F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) for c in numeric_cols])
null_counts_test.show()

+-----------+------------+-----------+----------+---------+-------------+----------------+-----------------+------------------+------------------+------------+----------------+----------------+-----------------+--------------------+------------------+--------------------+---------+------------+-------+-------+---------+--------------------+-------------------+-------------------------+------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+-------------+-----------------+-------------+-----------------+
|period_open|period_close|period_high|period_low|avg_close|period_return|price_volatility|return_volatility|avg_intraday_range|max_intraday_range|total_volume|avg_daily_volume|max_daily_volume|volume_volatility|volume_concentration|avg_volume_up_days|avg_volume_down_days|gap_count|avg_gap_size|max_gap|up_days|down_days|property_acquis

In [52]:
# inspect the missing values in train
train_df.show()

+------+---------------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+--------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+--------------------+-------------------+-------------------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+--------------------+-------------------+--------------------+-----------------+
|ticker|biweekly_period|period_start|period_end|period_open|period_close|period_high|period_low|         avg_close|       period_return|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range|

1. The missing values are in the **first period** because there is no prior data to compute the lagged features. You might choose to drop these rows or fill them with appropriate values (e.g., using the mean or mode of the feature). For simplicity, we will drop these rows here. 
2. The missing values in the **avg_volume** features exist because the stock price only went up/down during those periods, leading to undefined averages. We will impute those by filling them with 0.

In [53]:
train_df = train_df.filter(F.col("prev_period_return").isNotNull())
train_df = train_df.fillna(0)

In [54]:
# inspect the missing values in test
test_df.show()

+------+---------------+------------+----------+-----------+------------+-----------+----------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+--------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+--------------------+-------------------+-------------------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+--------------------+-------------------+-------------------+-----------------+
|ticker|biweekly_period|period_start|period_end|period_open|period_close|period_high|period_low|          avg_close|       period_return|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range

In [55]:
# We again fill the missing values for avg_volume features with 0
test_df = test_df.fillna(0)

### 2.3. Select Features and Assemble Feature Vector

In [56]:
# Feature selection
exclude_columns = ["biweekly_period", "period_start", "period_end", "next_announcement"]
feature_columns = [col for col in train_df.columns if col not in exclude_columns]
VA = VectorAssembler(inputCols=feature_columns, outputCol="features")

In [68]:
# Feature selection
from pyspark.ml.feature import VectorAssembler

# 1. Define columns to exclude from the final feature vector.
# CRITICAL CHANGE: The original string column "ticker" MUST be excluded 
# from this list, as it will be processed separately by StringIndexer/OneHotEncoder.
exclude_columns = ["biweekly_period", "period_start", "period_end", "next_announcement", "ticker"]

# 2. Identify all remaining numeric features.
# These columns will be passed directly to the VectorAssembler.
numeric_features = [col for col in train_df.columns if col not in exclude_columns]

# 3. Define the VectorAssembler (VA).
# It combines the list of numeric features plus the 'tickerVec' column 
# (which will be created by the OneHotEncoder step in your pipeline).
VA = VectorAssembler(inputCols=numeric_features + ["tickerVec"], outputCol="features")

# NOTE: Ensure your overall pipeline structure is:
# [StringIndexer] -> [OneHotEncoder] -> [VA] -> [StandardScaler] -> [LogisticRegression]

### 2.4. Scale Numeric Features

In [69]:
# define the standardizer (standard normal)
SS = StandardScaler(inputCol="features", outputCol="featuresNorm", withMean=False, withStd=True)

### 2.5 Logistic Regression

In [71]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder # Ensure these are imported
from pyspark.ml import Pipeline

# 1. Define Encoding Stages
# CORRECTED: Set handleInvalid to "keep" to prevent errors on unseen labels (FDY)
indexer = StringIndexer(inputCol="ticker", outputCol="tickerIndex", handleInvalid="keep")
encoder = OneHotEncoder(inputCol="tickerIndex", outputCol="tickerVec")

# 2. Define your existing stages (VA, SS, LR)
# VA is defined to use 'tickerVec'
# SS is defined to use 'features' (or 'featuresNorm')
# LR is defined to use 'featuresNorm' and 'next_announcement'

# 3. Build the Corrected Pipeline
# **CRITICAL FIX**: Include indexer and encoder BEFORE the VectorAssembler (VA)
lr_pipeline = Pipeline(stages=[indexer, encoder, VA, SS, LR])

# Model training (This should now work)
lr_model = lr_pipeline.fit(train_df)

In [72]:
# predictions
lr_predictions = lr_model.transform(test_df)

In [73]:
lr_predictions.printSchema()

root
 |-- ticker: string (nullable = true)
 |-- biweekly_period: string (nullable = true)
 |-- period_start: date (nullable = true)
 |-- period_end: date (nullable = true)
 |-- period_open: double (nullable = false)
 |-- period_close: double (nullable = false)
 |-- period_high: double (nullable = false)
 |-- period_low: double (nullable = false)
 |-- avg_close: double (nullable = false)
 |-- period_return: double (nullable = false)
 |-- price_volatility: double (nullable = false)
 |-- return_volatility: double (nullable = false)
 |-- avg_intraday_range: double (nullable = false)
 |-- max_intraday_range: double (nullable = false)
 |-- total_volume: double (nullable = false)
 |-- avg_daily_volume: double (nullable = false)
 |-- max_daily_volume: double (nullable = false)
 |-- volume_volatility: double (nullable = false)
 |-- volume_concentration: double (nullable = false)
 |-- avg_volume_up_days: double (nullable = false)
 |-- avg_volume_down_days: double (nullable = false)
 |-- gap_coun

In [74]:
# show predictions
lr_predictions.select("ticker", "period_start", "period_end", "next_announcement", "prediction").show()

+------+------------+----------+-----------------+----------+
|ticker|period_start|period_end|next_announcement|prediction|
+------+------------+----------+-----------------+----------+
|   FNM|  2024-05-06|2024-05-17|                0|       0.0|
|   FNM|  2024-05-21|2024-05-31|                0|       0.0|
|   FNM|  2024-06-03|2024-06-14|                0|       1.0|
|   FNM|  2024-06-17|2024-06-28|                0|       0.0|
|   FNM|  2024-07-02|2024-07-08|                0|       0.0|
|   FOM|  2024-11-18|2024-11-29|                0|       1.0|
|   FOM|  2024-12-02|2024-12-13|                0|       0.0|
|   FOM|  2024-12-16|2024-12-27|                0|       0.0|
|   FOM|  2024-12-30|2025-01-10|                0|       0.0|
|   FOM|  2025-01-13|2025-01-24|                0|       0.0|
|   FOM|  2025-01-27|2025-02-07|                0|       0.0|
|   FOM|  2025-02-10|2025-02-21|                0|       0.0|
|   FOM|  2025-02-24|2025-03-05|                0|       0.0|
|   FOM|

In [75]:
def calculate_classification_metrics(predictions, label_col, prediction_col):
    # calculate true positives, true negatives, false positives, false negatives
    tp = predictions.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).count()
    tn = predictions.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).count()
    fp = predictions.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).count()
    fn = predictions.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).count()

    # calculate accuracy
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    # calculate precision
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0  
    
    # calculate recall
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0.0  
    
    # calculate F1 measure
    f1_measure = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0.0  

    # print the metrics
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 measure: {f1_measure}")
    
    return accuracy, precision, recall, f1_measure

In [76]:
# get metrics
lr_accuracy, lr_precision, lr_recall, lr_f1_measure = calculate_classification_metrics(lr_predictions, 'next_announcement', 'prediction')

Accuracy: 0.961038961038961
Precision: 0.25
Recall: 1.0
F1 measure: 0.4


In [77]:
# get AUC
evaluator = BinaryClassificationEvaluator(labelCol="next_announcement", 
                                          rawPredictionCol="probability", 
                                          metricName="areaUnderROC")

lr_AUC = evaluator.evaluate(lr_predictions)
print(f"AUC: {lr_AUC:.4f}")

AUC: 0.9868


### 2.8 Random Forest

In [79]:
# define random forest model
RF = RandomForestClassifier(featuresCol="features", labelCol="next_announcement", numTrees=100)

# fit model on train
rf_pipeline = Pipeline(stages=[indexer, encoder, VA, RF])

# model training
rf_model = rf_pipeline.fit(train_df)

In [80]:
# predictions
rf_predictions = rf_model.transform(test_df)

In [81]:
# show predictions
rf_predictions.select("ticker", "period_start", "period_end", "next_announcement", "prediction").show()

+------+------------+----------+-----------------+----------+
|ticker|period_start|period_end|next_announcement|prediction|
+------+------------+----------+-----------------+----------+
|   FNM|  2024-05-06|2024-05-17|                0|       0.0|
|   FNM|  2024-05-21|2024-05-31|                0|       0.0|
|   FNM|  2024-06-03|2024-06-14|                0|       0.0|
|   FNM|  2024-06-17|2024-06-28|                0|       0.0|
|   FNM|  2024-07-02|2024-07-08|                0|       0.0|
|   FOM|  2024-11-18|2024-11-29|                0|       0.0|
|   FOM|  2024-12-02|2024-12-13|                0|       0.0|
|   FOM|  2024-12-16|2024-12-27|                0|       0.0|
|   FOM|  2024-12-30|2025-01-10|                0|       0.0|
|   FOM|  2025-01-13|2025-01-24|                0|       0.0|
|   FOM|  2025-01-27|2025-02-07|                0|       0.0|
|   FOM|  2025-02-10|2025-02-21|                0|       0.0|
|   FOM|  2025-02-24|2025-03-05|                0|       0.0|
|   FOM|

In [82]:
# get metrics
rf_accuracy, rf_precision, rf_recall, rf_f1_measure = calculate_classification_metrics(rf_predictions, 'next_announcement', 'prediction')

Accuracy: 0.987012987012987
Precision: 0
Recall: 0.0
F1 measure: 0.0


This model simply predicts the minority class all the time based on Precision and Recall being 0.0 (check results of confusion matrix)

In [83]:
# get AUC
rf_AUC = evaluator.evaluate(rf_predictions)
print(f"AUC: {rf_AUC:.4f}")

AUC: 0.3947


### 2.9 Gradient-Boosted Tree Classifier

In [84]:
# define boosting model
GBT = GBTClassifier(featuresCol="featuresNorm", labelCol="next_announcement")

# fit model on train
gbt_pipeline = Pipeline(stages=[indexer, encoder, VA, SS, GBT])

# model training
gbt_model = gbt_pipeline.fit(train_df)

In [85]:
# predictions
gbt_predictions = gbt_model.transform(test_df)

In [86]:
# show predictions
gbt_predictions.select("ticker", "period_start", "period_end", "next_announcement", "prediction").show()

+------+------------+----------+-----------------+----------+
|ticker|period_start|period_end|next_announcement|prediction|
+------+------------+----------+-----------------+----------+
|   FNM|  2024-05-06|2024-05-17|                0|       0.0|
|   FNM|  2024-05-21|2024-05-31|                0|       0.0|
|   FNM|  2024-06-03|2024-06-14|                0|       1.0|
|   FNM|  2024-06-17|2024-06-28|                0|       1.0|
|   FNM|  2024-07-02|2024-07-08|                0|       0.0|
|   FOM|  2024-11-18|2024-11-29|                0|       0.0|
|   FOM|  2024-12-02|2024-12-13|                0|       0.0|
|   FOM|  2024-12-16|2024-12-27|                0|       0.0|
|   FOM|  2024-12-30|2025-01-10|                0|       0.0|
|   FOM|  2025-01-13|2025-01-24|                0|       0.0|
|   FOM|  2025-01-27|2025-02-07|                0|       0.0|
|   FOM|  2025-02-10|2025-02-21|                0|       0.0|
|   FOM|  2025-02-24|2025-03-05|                0|       0.0|
|   FOM|

In [87]:
# get metrics
gbt_accuracy, gbt_precision, gbt_recall, gbt_f1_measure = calculate_classification_metrics(gbt_predictions, 'next_announcement', 'prediction')

Accuracy: 0.948051948051948
Precision: 0.0
Recall: 0.0
F1 measure: 0.0


In [88]:
# get AUC
gbt_AUC = evaluator.evaluate(gbt_predictions)
print(f"AUC: {gbt_AUC:.4f}")

AUC: 0.7105


### 2.10 Overview

In [89]:
# check which algorithm is the best
print("LOGISTIC REGRESSION:")
print('  ACCURACY   : %g' % lr_accuracy)
print('  PRECISION  : %g' % lr_precision)
print('  RECALL     : %g' % lr_recall)
print('  AUC        : %g' % lr_AUC)
print('  F1         : %g' % lr_f1_measure)
print("------------------")
print("RANDOM FOREST:")
print('  ACCURACY   : %g' % rf_accuracy)
print('  PRECISION  : %g' % rf_precision)
print('  RECALL     : %g' % rf_recall)
print('  AUC        : %g' % rf_AUC)
print('  F1         : %g' % rf_f1_measure)
print("------------------")
print("GRADIENT BOOSTED TREES:")
print('  ACCURACY   : %g' % gbt_accuracy)
print('  PRECISION  : %g' % gbt_precision)
print('  RECALL     : %g' % gbt_recall)
print('  AUC        : %g' % gbt_AUC)
print('  F1         : %g' % gbt_f1_measure)

LOGISTIC REGRESSION:
  ACCURACY   : 0.961039
  PRECISION  : 0.25
  RECALL     : 1
  AUC        : 0.986842
  F1         : 0.4
------------------
RANDOM FOREST:
  ACCURACY   : 0.987013
  PRECISION  : 0
  RECALL     : 0
  AUC        : 0.394737
  F1         : 0
------------------
GRADIENT BOOSTED TREES:
  ACCURACY   : 0.948052
  PRECISION  : 0
  RECALL     : 0
  AUC        : 0.710526
  F1         : 0


In [90]:
# Stop the Spark session
spark.stop()
print("Spark session stopped successfully!")
print("🎉 DEMO 2 COMPLETED! Ready for DEMO 3.")

Spark session stopped successfully!
🎉 DEMO 2 COMPLETED! Ready for DEMO 3.


**Note:** As expected, with such a small dataset the models will likely not perform well, predicting the majority class all the time. For the assignment, you should use datasets of multiple companies to increase the data size and improve model performance. When combining data from multiple companies, ensure that the variables are standardized across companies to maintain consistency in the features used for modeling. For example, stock prices and volumes should be normalized or scaled appropriately to account for differences in price levels between companies. You can also include company-specific features, such as market capitalization, to help the model differentiate between companies.

### Future Steps and Potential Improvements

**Autoregressive models**: A base model where you predict the next_announcement based on lagged values of the target variable.

**Feature engineering**: Use different lags or create additional features that capture long-term trends or anomalies in stock price movements/volume.

**Model Tuning**: Implement cross-validation techniques to fine-tune hyperparameters and improve model performance.

**Class imbalance handling**: Experiment with sampling or using class weights to address the imbalance in the target variable.

**Data Aggregation**: Explore the possibility of aggregating data of multiple companies to have more points for training the model. This way, you can add company-specific indicators as features, which may enhance the model's predictive power. Part of the assignment is to look how generalizable your model is across different stocks. Also make sure features are normalized properly across different stocks (such as stock price, volume, etc.) to ensure better generalizability.